# Notebook 6 — Decision Trees and Random Forests


## Learning objectives

- Explain the recursive-splitting algorithm and the impurity measures (MSE, Gini, entropy).
- Visualise a decision tree's first few splits.
- Diagnose overfitting in a depth sweep.
- Build a random forest, understand bagging and random feature subsampling.
- Use `TimeSeriesSplit` cross-validation to tune hyper-parameters honestly.
- Inspect built-in feature importances and recognise their limitations.

## 6.1 The need for non-linear models

Linear regression and logistic regression assume the relationship between features and target is a linear (or log-linear) function plus noise. Real atmospheric processes are full of *non-linearities*: a wind speed of 5 m/s might disperse pollution; 15 m/s might churn up dust. Temperature *increases* photochemical $O_{3}$ production until a saturation point. **Decision trees** capture such effects without any prior commitment to a functional form.

## 6.2 Decision trees

### 6.2.1 Anatomy

A **decision tree** is a binary tree in which:

- each *internal node* tests one feature against one threshold (`if PM2.5_lag_1 < 80 ...`);
- each *leaf* contains a constant prediction (a number for regression, a class label for classification).

To predict, follow the root-to-leaf path implied by the test outcomes for the input row, then return the leaf's constant.

### 6.2.2 Training: the recursive splitting algorithm

Trees are grown greedily. Starting with all training rows at the root:

1. Consider every feature and every candidate threshold.
2. Evaluate the *split quality* for each (feature, threshold) pair using a chosen impurity measure.
3. Select the split that maximally reduces impurity.
4. Recurse on the two resulting child nodes until a stopping criterion is met (max depth, minimum samples per leaf, no further impurity reduction possible).

For regression the impurity at a node $S$ is the mean squared error within the node:

$$\text{MSE}(S) = \frac{1}{|S|} \sum_{i \in S} (y_{i} - \bar{y}_{S})^{2}.$$

A split into left/right children $S_{L}, S_{R}$ has post-split impurity

$$\frac{|S_{L}|}{|S|} \text{MSE}(S_{L}) + \frac{|S_{R}|}{|S|} \text{MSE}(S_{R}),$$

and the best split is the one minimising this quantity over all (feature, threshold) candidates.

For classification the standard impurities are **Gini** and **entropy**:

$$\text{Gini}(S) = 1 - \sum_{k=1}^{K} p_{k}^{2}, \qquad \text{Entropy}(S) = -\sum_{k=1}^{K} p_{k} \log p_{k}.$$

### 6.2.3 Strengths and weaknesses

**Strengths.** No scaling needed (splits are scale-invariant). Handles mixed numeric / categorical features. Explicitly non-linear. Produces a fully interpretable rule set.

**Weaknesses.** A single tree is *high variance*: small changes in the training set produce very different trees. Trees overfit unless regularised (depth, leaf count, leaf size). Tree splits cannot exploit the *circular* nature of cyclic features the way $\sin/\cos$ encodings do; they will instead carve up the integer hour with axis-aligned cuts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

DATA_DIR = Path('data')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

csv = DATA_DIR / 'air_quality_nongzhanguan_forecast.csv'
if not csv.exists():
    csv = DATA_DIR / 'air_quality_aotizhongxin_pm25_forecast.csv'
df = pd.read_csv(csv, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)

lag_cols = [c for c in df.columns if '_lag_' in c]
CYCLIC = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']
METEO  = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
NUMERIC = ['PM10', 'SO2', 'NO2', 'CO', 'O3'] + METEO + CYCLIC + lag_cols
TARGET = 'target_pm25_h1'

cut = int(0.8 * len(df))
train_df, test_df = df.iloc[:cut], df.iloc[cut:]
print(f'Train rows: {len(train_df):,}  Test rows: {len(test_df):,}')

### A single decision tree of depth 3

For a tree of depth 3 we can render the full split structure. The root usually splits on `PM2.5_lag_1` — the strongest single predictor.

In [ ]:
# Trees don't need scaling — but we keep one-hot for wd via ColumnTransformer.
pre = ColumnTransformer([
    ('num', 'passthrough', NUMERIC),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd']),
])
tree3 = Pipeline([('pre', pre), ('tree', DecisionTreeRegressor(max_depth=3, random_state=0))])
tree3.fit(train_df[NUMERIC + ['wd']], train_df[TARGET])
mae = mean_absolute_error(test_df[TARGET], tree3.predict(test_df[NUMERIC + ['wd']]))
r2  = r2_score(test_df[TARGET], tree3.predict(test_df[NUMERIC + ['wd']]))
print(f'Depth-3 tree:  MAE={mae:.2f}  R2={r2:.3f}')

In [ ]:
# Render the depth-3 tree
feat_names = NUMERIC + list(tree3.named_steps['pre'].named_transformers_['cat'].get_feature_names_out(['wd']))
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(tree3.named_steps['tree'], feature_names=feat_names, filled=True,
          fontsize=8, max_depth=3, ax=ax, impurity=False, precision=1)
plt.title('Decision tree, depth 3 — first split is on the strongest lag')
plt.tight_layout(); plt.show()

Each box shows the splitting feature and threshold, the number of samples that reach that node, and the predicted value (`value =`). Reading from root to leaf gives a human-readable rule.

### Overfitting: depth sweep

A single tree with no depth limit will memorise the training set. The classic overfit profile is **train MAE keeps falling with depth, test MAE bottoms out and starts rising**.

In [ ]:
depths = [1, 3, 5, 10, 20, None]
rows = []
for d in depths:
    pipe = Pipeline([('pre', pre), ('tree', DecisionTreeRegressor(max_depth=d, random_state=0))])
    pipe.fit(train_df[NUMERIC + ['wd']], train_df[TARGET])
    train_mae = mean_absolute_error(train_df[TARGET], pipe.predict(train_df[NUMERIC + ['wd']]))
    test_mae  = mean_absolute_error(test_df[TARGET],  pipe.predict(test_df[NUMERIC + ['wd']]))
    rows.append({'max_depth': str(d), 'train_MAE': train_mae, 'test_MAE': test_mae})
overfit_df = pd.DataFrame(rows).set_index('max_depth')
overfit_df.plot(kind='bar', figsize=(9, 4), color=['lightgray', 'tab:red'])
plt.title('Single decision tree: train vs test MAE — classic overfit profile')
plt.ylabel('MAE'); plt.tight_layout(); plt.show()
overfit_df

## 6.3 Random forests

### 6.3.1 Bagging

A random forest mitigates the variance of a single tree by *averaging* many trees, each trained on a slightly different version of the data. The general technique is **bagging** (bootstrap aggregating):

1. Draw $B$ *bootstrap samples* of the training set, each obtained by sampling $N$ rows with replacement.
2. Train one tree on each sample.
3. For regression, average the $B$ tree predictions; for classification, vote.

Averaging reduces variance because uncorrelated errors of individual trees partially cancel:

$$\text{Var}\left(\frac{1}{B}\sum_{b=1}^{B} f_{b}(\mathbf{x})\right) = \frac{1}{B^{2}} \sum_{b=1}^{B} \text{Var}(f_{b}) + \frac{1}{B^{2}}\sum_{b \neq c} \text{Cov}(f_{b}, f_{c}).$$

If the trees were independent, the second term would vanish and total variance would be $\text{Var}(f_{b}) / B$. They are not independent — the trees see overlapping data — so the variance reduction is partial.

### 6.3.2 Random feature subsampling

In a *random forest*, at each split we consider only a *random subset* of features rather than all $d$. Typical defaults are: $\sqrt{d}$ for classification, $d/3$ for regression. This forces individual trees to use different features and reduces $\text{Cov}(f_{b}, f_{c})$.

The combination — bootstrap rows + random features per split + averaged predictions — defines the random forest of Breiman (2001).

### 6.3.3 Out-of-bag evaluation

Each bootstrap sample omits roughly $(1-1/N)^{N} \approx 37\%$ of the training rows. Those *out-of-bag* rows can be used as a free validation set for the corresponding tree, giving an honest performance estimate without a separate held-out set. Useful when data is scarce. **For time series it is not equivalent to a chronological test set** — see misconceptions below.

In [ ]:
rf = Pipeline([('pre', pre),
               ('rf', RandomForestRegressor(n_estimators=200, max_depth=None,
                                              min_samples_leaf=5, random_state=0, n_jobs=-1))])
rf.fit(train_df[NUMERIC + ['wd']], train_df[TARGET])
pred_rf = rf.predict(test_df[NUMERIC + ['wd']])
print(f'RandomForest:  MAE={mean_absolute_error(test_df[TARGET], pred_rf):.2f}  R2={r2_score(test_df[TARGET], pred_rf):.3f}')

lr = Pipeline([('pre', ColumnTransformer([
                  ('num', StandardScaler(), NUMERIC),
                  ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd'])])),
                ('lr', LinearRegression())])
lr.fit(train_df[NUMERIC + ['wd']], train_df[TARGET])
pred_lr = lr.predict(test_df[NUMERIC + ['wd']])
print(f'LinearReg   :  MAE={mean_absolute_error(test_df[TARGET], pred_lr):.2f}  R2={r2_score(test_df[TARGET], pred_lr):.3f}')

## 6.4 Time-based cross-validation

We must combine random forests with the time-based split rules of Notebook 4. `TimeSeriesSplit` produces forward-only folds; each fold's train set strictly precedes its test set; the train set grows over folds (expanding window).

The project uses `TimeSeriesSplit(n_splits=5)` with `scoring='neg_mean_absolute_error'`:

```python
model = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=TimeSeriesSplit(n_splits=5),
    n_jobs=-1,
)
```

`scoring='neg_mean_absolute_error'` is a `sklearn` quirk: optimisation is by maximisation, so MAE is negated to make "less wrong" mean "more positive".

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
fig, ax = plt.subplots(figsize=(12, 3))
for i, (tr, te) in enumerate(tscv.split(train_df)):
    ax.barh(i, len(tr), color='steelblue', label='train' if i == 0 else None)
    ax.barh(i, len(te), left=len(tr), color='tab:red', label='test' if i == 0 else None)
ax.set_yticks(range(5)); ax.set_yticklabels([f'fold {i+1}' for i in range(5)])
ax.set_xlabel('row index'); ax.set_title('TimeSeriesSplit folds (expanding window)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Tune RF max_depth via TimeSeriesSplit
grid = GridSearchCV(
    estimator=Pipeline([('pre', pre),
                         ('rf', RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1))]),
    param_grid={'rf__max_depth': [10, 20, None]},
    cv=TimeSeriesSplit(n_splits=3),
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
grid.fit(train_df[NUMERIC + ['wd']], train_df[TARGET])
print('Best CV params:', grid.best_params_)
print(f'Best CV MAE   : {-grid.best_score_:.3f}')
pred_best = grid.predict(test_df[NUMERIC + ['wd']])
print(f'Test MAE      : {mean_absolute_error(test_df[TARGET], pred_best):.2f}')
print(f'Test R²       : {r2_score(test_df[TARGET], pred_best):.3f}')

## 6.5 Real-world application — random forest at *Nongzhanguan*

For $PM_{2.5}$ forecasting at *Nongzhanguan* on `baseline_plus_lags`, a random forest with `max_depth=20`, `min_samples_leaf=5`, `n_estimators=200` typically achieves MAE within a few per cent of the gradient-boosted machine, but at considerably higher inference cost (200 trees of depth 20 vs 200 trees of depth 3 for GBM). Random forests are an excellent first non-linear model: trivially parallelisable, robust to hyper-parameter mis-tuning, and informative through their *feature-importance* outputs (Notebook 8).

### Built-in feature importance

In [ ]:
rf_model = grid.best_estimator_.named_steps['rf']
feat_names = NUMERIC + list(grid.best_estimator_.named_steps['pre']
                            .named_transformers_['cat'].get_feature_names_out(['wd']))
imp = pd.Series(rf_model.feature_importances_, index=feat_names).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 7))
imp.head(20).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top-20 RF feature importances')
plt.tight_layout(); plt.show()

Almost certainly `PM2.5_lag_1` dominates. **Caveats**:

- Tree feature importance is **biased toward high-cardinality features** (those with many distinct values).
- It **splits the importance among correlated columns** — drop one of two correlated features and the surviving feature's reported importance will jump.

We address both by computing **permutation importance** in Notebook 8.

## 6.6 Common misconceptions

- **"Random forests don't overfit."** They overfit *less* than single trees, but uncapped depth / leaf size still memorises. A forest trained on $N$ rows can memorise the training set in extreme regimes.
- **"More trees always help."** Beyond a few hundred trees, additional trees mainly add cost without adding accuracy — variance reduction has plateaued. The project caps `n_estimators` at 200 by default.
- **"Trees handle cyclic features automatically."** They handle numeric features non-linearly, but they cannot represent the wrap-around between hour 23 and hour 0 with axis-aligned splits. Always provide $(\sin, \cos)$ encodings.
- **"Out-of-bag accuracy replaces a test set."** It is a useful internal estimate, but it is computed on rows that are *contemporaneous* with training rows. For time-series problems it does *not* replace a chronological test set.

---
## Exercises

### Exercise 6.1 — `min_samples_leaf` sweep

*Hint: hold `max_depth=None` and sweep `min_samples_leaf in [1, 5, 20, 100]`. Larger leaves regularise the forest. Where does test MAE turn around?*

In [ ]:
# EXERCISE 6.1
# for leaf in [1, 5, 20, 100]:
#     ...
# TODO: line plot of test MAE vs min_samples_leaf.


### Exercise 6.2 — h = 12 challenge

*Hint: re-run the RF vs LR comparison with `TARGET = 'target_pm25_h12'`. Does the RF advantage over linear regression grow with horizon? It usually does — at $h=12$ the autoregressive signal is weaker so non-linear interactions matter more.*

In [ ]:
# EXERCISE 6.2
# TODO: rerun the RF vs LR comparison at h=12 and plot MAE for both horizons side-by-side.


### Exercise 6.3 — out-of-bag estimate

*Hint: pass `oob_score=True, bootstrap=True` to `RandomForestRegressor`. Compare `model.oob_score_` to your test $R^2$. Why is OOB unreliable for time series? (Because OOB rows are interleaved with in-bag rows in time, so they leak temporal information.)*

In [ ]:
# EXERCISE 6.3
# rf_oob = RandomForestRegressor(n_estimators=200, oob_score=True, bootstrap=True, random_state=0, n_jobs=-1)
# TODO: fit on train, compare rf_oob.oob_score_ to chronological test R².


## 6.7 Chapter summary

- A decision tree partitions feature space by axis-aligned splits chosen greedily to minimise within-node impurity (MSE for regression, Gini/entropy for classification).
- Single trees are high-variance; random forests bag many trees on bootstrap samples, with random feature subsampling at each split, to reduce variance and decorrelate predictors.
- Random forests are scale-invariant, handle mixed feature types, and provide free out-of-bag estimates and feature-importance scores.
- For time-ordered atmospheric data, cross-validation must respect chronology — `TimeSeriesSplit(n_splits=5)` is the default cross-validator inside `GridSearchCV`.
- Random forests are a strong non-linear baseline; gradient boosting (Notebook 7) typically squeezes out a few more percentage points of accuracy at lower inference cost per tree.

**Next:** Notebook 7 introduces **boosting** — sequential rather than parallel ensembles — and the gradient-boosting machine that wins 15 of 18 forecasting combinations on this dataset.